In [1]:
import os, json, shutil
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score, classification_report

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html  from .autonotebook import tqdm as notebook_tqdm

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA RTX A4000


In [2]:
ROOT = os.path.abspath(os.path.join('..', '..'))
BASE = os.path.join(ROOT, 'Nastaliq', 'Fake News Detection')

# Domain A: Multi-Topic News (build from Train/ and Test/ folders)
def build_domain_a():
    a_train_dir = os.path.join(BASE, 'Domain_A_Multi_Topic_News', 'Train')
    a_test_dir = os.path.join(BASE, 'Domain_A_Multi_Topic_News', 'Test')

    def process_folder(folder_path, label):
        texts = []
        for fname in os.listdir(folder_path):
            if fname.endswith('.txt'):
                with open(os.path.join(folder_path, fname), 'r', encoding='utf-8') as f:
                    texts.append(f.read())
        return pd.DataFrame({'text': texts, 'label': label})

    a_train_fake = process_folder(os.path.join(a_train_dir, 'Fake'), 0)
    a_train_real = process_folder(os.path.join(a_train_dir, 'Real'), 1)
    a_test_fake = process_folder(os.path.join(a_test_dir, 'Fake'), 0)
    a_test_real = process_folder(os.path.join(a_test_dir, 'Real'), 1)

    a_train = pd.concat([a_train_fake, a_train_real], ignore_index=True)
    a_test = pd.concat([a_test_fake, a_test_real], ignore_index=True)

    # Shuffle
    a_train = a_train.sample(frac=1, random_state=42).reset_index(drop=True)
    a_test = a_test.sample(frac=1, random_state=42).reset_index(drop=True)

    return a_train, a_test

dfa_train, dfa_test = build_domain_a()

# Domain B: Ax-to-Grind
dfb_train = pd.read_csv(os.path.join(BASE, 'Domain_B_Ax_to_Grind', 'train.csv'), encoding='utf-8-sig')
dfb_test = pd.read_csv(os.path.join(BASE, 'Domain_B_Ax_to_Grind', 'test.csv'), encoding='utf-8-sig')

# Domain C: General Pakistani News
# Merge Fake News 12166.xlsx (label=0) and Final True News-11012.xlsx (label=1)
fake_file = os.path.join(BASE, 'Domain_C_General_Pakistani_News', 'Fake News 12166.xlsx')
real_file = os.path.join(BASE, 'Domain_C_General_Pakistani_News', 'Final True News-11012.xlsx')

dfc_fake = pd.read_excel(fake_file, engine='openpyxl')
dfc_real = pd.read_excel(real_file, engine='openpyxl')

# Ensure text column exists
if 'text' not in dfc_fake.columns:
    if 'content' in dfc_fake.columns:
        dfc_fake = dfc_fake.rename(columns={'content': 'text'})
    elif 'news_content' in dfc_fake.columns:
        dfc_fake = dfc_fake.rename(columns={'news_content': 'text'})

if 'text' not in dfc_real.columns:
    if 'content' in dfc_real.columns:
        dfc_real = dfc_real.rename(columns={'content': 'text'})
    elif 'news_content' in dfc_real.columns:
        dfc_real = dfc_real.rename(columns={'news_content': 'text'})

dfc_fake['label'] = 0
dfc_real['label'] = 1

# Combine and split 80/20
dfc_combined = pd.concat([dfc_fake, dfc_real], ignore_index=True)

# Create train/test split (80/20)
dfc_shuffled = dfc_combined.sample(frac=1, random_state=42).reset_index(drop=True)
dfc_split_idx = int(0.8 * len(dfc_shuffled))
dfc_train = dfc_shuffled.iloc[:dfc_split_idx].reset_index(drop=True)
dfc_test = dfc_shuffled.iloc[dfc_split_idx:].reset_index(drop=True)

# Domain D: Fact-Checking Platform
dfd_train = pd.read_excel(os.path.join(BASE, 'Domain_D_Fact_Checking_Platform', 'train.csv'), engine='openpyxl')
dfd_test = pd.read_excel(os.path.join(BASE, 'Domain_D_Fact_Checking_Platform', 'test.csv'), engine='openpyxl')

# Ensure text column exists
if 'text' not in dfd_train.columns:
    if 'content' in dfd_train.columns:
        dfd_train = dfd_train.rename(columns={'content': 'text'})
    elif 'news_content' in dfd_train.columns:
        dfd_train = dfd_train.rename(columns={'news_content': 'text'})

if 'text' not in dfd_test.columns:
    if 'content' in dfd_test.columns:
        dfd_test = dfd_test.rename(columns={'content': 'text'})
    elif 'news_content' in dfd_test.columns:
        dfd_test = dfd_test.rename(columns={'news_content': 'text'})

# Map Real/Unreal to 1/0 labels
if 'label' not in dfd_train.columns and 'Label' in dfd_train.columns:
    dfd_train = dfd_train.rename(columns={'Label': 'label'})
if 'label' not in dfd_test.columns and 'Label' in dfd_test.columns:
    dfd_test = dfd_test.rename(columns={'Label': 'label'})

if 'Unreal' in dfd_train['label'].unique():
    dfd_train['label'] = dfd_train['label'].map({'Unreal': 0, 'Real': 1})
if 'Unreal' in dfd_test['label'].unique():
    dfd_test['label'] = dfd_test['label'].map({'Unreal': 0, 'Real': 1})

print('Domain sizes (train / test):')
for name, tr, te in [
    ('A_Multi_Topic_News',       dfa_train, dfa_test),
    ('B_Ax_to_Grind',              dfb_train, dfb_test),
    ('C_General_Pakistani_News',        dfc_train, dfc_test),
    ('D_Fact_Checking_Platform',       dfd_train, dfd_test),
]:
    print(f'  {name}: train={len(tr)}  test={len(te)}  labels={tr["label"].value_counts().to_dict()}')


Domain sizes (train / test):
  A_Multi_Topic_News: train=800  test=200  labels={0: 400, 1: 400}
  B_Ax_to_Grind: train=8020  test=2005  labels={0: 4014, 1: 4006}
  C_General_Pakistani_News: train=7996  test=1999  labels={0: 3998, 1: 3998}
  D_Fact_Checking_Platform: train=10678  test=2670  labels={0: 5339, 1: 5339}


In [3]:
XLM_MODEL_ID = 'xlm-roberta-base'
BERT_MODEL_ID = 'bert-base-multilingual-cased'

NUM_LABELS = 2
MAX_LEN = 128
MAX_TRAIN = 8000
EPOCHS = 5
PATIENCE = 2
BATCH_TRAIN = 16
BATCH_EVAL = 32
LR = 2e-5

RESULTS_BASE  = os.path.join(ROOT, 'results', 'T1_Nastaliq_FND')
os.makedirs(RESULTS_BASE, exist_ok=True)

domains = [
    ('A_Multi_Topic_News',       dfa_train, dfa_test),
    ('B_Ax_to_Grind',              dfb_train, dfb_test),
    ('C_General_Pakistani_News',        dfc_train, dfc_test),
    ('D_Fact_Checking_Platform',       dfd_train, dfd_test),
]


In [ ]:
def cap_dataset(df, max_samples=MAX_TRAIN):
    if len(df) <= max_samples:
        return df
    capped = df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(
            min(len(x), round(max_samples * len(x) / len(df))),
            random_state=42
        )
    )
    return capped.sample(frac=1, random_state=42).reset_index(drop=True)


class UrduDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.labels = df['label'].astype(int).tolist()
        self.enc = tokenizer(
            df['text'].astype(str).tolist(),
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds)
    }


def run_one(model_id, model_label, src_name, train_df, tgt_name, test_df):
    out_dir     = os.path.join(RESULTS_BASE, model_label)
    os.makedirs(out_dir, exist_ok=True)
    result_path = os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')

    if os.path.exists(result_path):
        print(f'  [SKIP] {src_name} -> {tgt_name} already done.')
        with open(result_path, encoding='utf-8') as f:
            return json.load(f)['macro_f1'], None

    run_type = 'IN-DOMAIN' if src_name == tgt_name else 'CROSS-DOMAIN'
    capped   = cap_dataset(train_df)
    print(f'\n  [{run_type}] Train: {src_name} ({len(capped)}) -> Test: {tgt_name} ({len(test_df)})')

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_LABELS)

    val_df    = capped.sample(max(int(len(capped) * 0.1), 1), random_state=42)
    train_sub = capped.drop(val_df.index)

    ckpt_dir = os.path.join(out_dir, f'_ckpt_{src_name}_{tgt_name}')

    args = TrainingArguments(
        output_dir                  = ckpt_dir,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_TRAIN,
        per_device_eval_batch_size  = BATCH_EVAL,
        learning_rate               = LR,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'macro_f1',
        greater_is_better           = True,
        logging_steps               = 50,
        fp16                        = torch.cuda.is_available(),
        report_to                   = 'none',
        save_total_limit            = 1,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = UrduDataset(train_sub, tokenizer),
        eval_dataset    = UrduDataset(val_df, tokenizer),
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    )
    trainer.train()

    preds_out = trainer.predict(UrduDataset(test_df, tokenizer))
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = preds_out.label_ids

    macro_f1 = f1_score(labels, preds, average='macro')
    accuracy = accuracy_score(labels, preds)

    result = {
        'task': 'T1_Nastaliq_FND', 'model': model_label, 'model_id': model_id,
        'source': src_name, 'target': tgt_name, 'type': run_type,
        'train_size': len(capped), 'test_size': len(test_df),
        'macro_f1': round(macro_f1, 4), 'accuracy': round(accuracy, 4),
        'classification_report': classification_report(labels, preds, output_dict=True)
    }
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    if os.path.exists(ckpt_dir):
        shutil.rmtree(ckpt_dir)

    print(f'  macro-F1={macro_f1:.4f}  accuracy={accuracy:.4f}')
    return macro_f1, trainer

print('Helpers loaded. Ready to run experiments.')

print('=== XLM-R | Source: A_Multi_Topic_News ===')
xlmr_results = globals().get('xlmr_results', {})

src_name, train_df, _ = domains[0]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource A done.')

print('=== XLM-R | Source: B_Ax_to_Grind ===')
src_name, train_df, _ = domains[1]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource B done.')

print('=== XLM-R | Source: C_General_Pakistani_News ===')
src_name, train_df, _ = domains[2]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource C done.')

print('=== XLM-R | Source: D_Fact_Checking_Platform ===')
src_name, train_df, _ = domains[3]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource D done.')

print('=== XLM-R | Source: All Domains Done ===')
print('XLM-R — all 16 runs complete.')
print('Results so far:')
for (s, t), f1 in sorted(xlmr_results.items()):
    tag = 'IN ' if s == t else 'X  '
    print(f'  [{tag}] {s} -> {t}: {f1:.4f}')

print('=== mBERT | Source: A_Multi_Topic_News ===')
mbert_results = globals().get('mbert_results', {})

src_name, train_df, _ = domains[0]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource A done.')

print('=== mBERT | Source: B_Ax_to_Grind ===')
src_name, train_df, _ = domains[1]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource B done.')

print('=== mBERT | Source: C_General_Pakistani_News ===')
src_name, train_df, _ = domains[2]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource C done.')

print('=== mBERT | Source: D_Fact_Checking_Platform ===')
src_name, train_df, _ = domains[3]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource D done.')

print('=== mBERT | Source: All Domains Done ===')
print('mBERT — all 16 runs complete.')
print('Results so far:')
for (s, t), f1 in sorted(mbert_results.items()):
    tag = 'IN ' if s == t else 'X  '
    print(f'  [{tag}] {s} -> {t}: {f1:.4f}')